# Algoritmo Genético para Resolución de Laberintos: Flujo Secuencial (CPU)

Este cuaderno repasa y defiende la teoría científica, el modelo matemático y el código secuencial de nuestro proyecto de optimización.

---

## 1. Paradigma Evolutivo vs. Algoritmos Tradicionales (Dijkstra, A*)

### ¿Por qué usar un Algoritmo Genético (AG)?
Los algoritmos tradicionales de búsqueda en grafos, como **Dijkstra** o **A***, son eficientes para encontrar el camino más corto en entornos conocidos y estáticos, siempre que el problema se modele sobre un grafo de adyacencias directo. Sin embargo:
1. **Modelado Basado en Agentes**: En nuestro problema, el robot se comporta como un autómata con una secuencia discreta de acciones fijas (avanzar, girar, detenerse). No buscamos simplemente un camino espacial, sino una **secuencia óptima de comandos** (control policy) de longitud fija $n$.
2. **Restricciones de Control y Complejidad**: Optimizar variables como el consumo energético (longitud de camino $\tau$), evitar choques, detener el movimiento inmediatamente al llegar a la meta (operación $Q$) y penalizar giros inútiles requiere evaluar funciones de costo altamente no lineales y no diferenciables. Dijkstra y A* no pueden incorporar estas penalizaciones dinámicas fácilmente en su función de costo sin romper la admisibilidad de la heurística.
3. **Espacio de Búsqueda Discreto**: El espacio de búsqueda está constituido por todas las posibles secuencias de longitud $n$ formadas por el alfabeto $A = \{H, A, M, Q\}$, lo que da un tamaño del espacio de búsqueda de $|A|^n = 4^n$. Para $n = 1000$, el espacio es de $4^{1000} \approx 10^{602}$ combinaciones, haciéndolo intratable para búsquedas exhaustivas.

### Exploración vs. Explotación en el AG
El AG balancea de forma adaptativa estas dos fuerzas:
* **Exploración (Diversificación)**: Introducida principalmente por el operador de **Mutación**. Permite explorar zonas no visitadas del espacio de búsqueda discreto modificando genes de forma aleatoria e independiente con probabilidad $pm$, rompiendo óptimos locales.
* **Explotación (Intensificación)**: Introducida por la **Selección** y el **Cruce**. El cruce monopunto toma segmentos exitosos (sub-rutas útiles que evitan choques o se acercan a la meta) de padres viables y los recombina para formar descendientes con mejor aptitud, guiando el proceso hacia la convergencia.

## 2. Teoría del Autómata (DFD Latente)

El comportamiento del robot en el laberinto se modela matemáticamente como un **Autómata Finito Determinista (DFA)** latente, donde el cromosoma actúa como la cinta de entrada que guía las transiciones del sistema.

### Definición Formal del Autómata:
El sistema se define por la tupla:
$$M = (S, \Sigma, \delta, s_0, F)$$

Donde:
* $S = \{(x, y, d)\}$ es el conjunto de estados discretos, donde $(x, y)$ representa la posición del robot en la cuadrícula y $d \in \{N, E, S, O\}$ es su orientación cardinal.
* $\Sigma = \{H, A, M, Q\}$ es el alfabeto de entrada (genes del cromosoma).
* $s_0 = (x_{inicio}, y_{inicio}, S)$ es el estado inicial (el robot arranca en la coordenada de inicio mirando al Sur).
* $F = \{(x_{meta}, y_{meta}, d)\}$ es el estado de aceptación (la meta del laberinto).
* $\delta: S \times \Sigma \to S$ es la función de transición de estados que opera sobre la matriz discreta del laberinto $L \in \{0, 1, 2, X\}^{m \times r}$:
  * $\delta((x, y, d), H) = (x, y, d \text{ girado } 90^\circ \text{ sentido horario})$
  * $\delta((x, y, d), A) = (x, y, d \text{ girado } 90^\circ \text{ sentido antihorario})$
  * $\delta((x, y, d), M) = (x + dx, y + dy, d)$ si la celda es transitable (no es muro `'X'`), de lo contrario se produce un choque y el estado se mantiene en $(x, y, d)$.
  * $\delta((x, y, d), Q) = (x, y, d)$ (Detención y reposo).

## 3. Carga del Entorno y Validaciones (`parser_csv.py`)

La carga del laberinto verifica estrictamente las restricciones del perímetro y el entorno para asegurar la integridad de la simulación:

In [ ]:
# Fragmento de código de carga y validaciones en parser_csv.py
import numpy as np
import csv
import sys

def cargar_laberinto(ruta_archivo):
    matriz_temporal = []
    with open(ruta_archivo, mode="r", encoding="utf-8") as archivo:
        lector_csv = csv.reader(archivo)
        for fila in lector_csv:
            fila_limpia = [celda.strip() for celda in fila]
            # Validar símbolos
            for celda in fila_limpia:
                if celda not in ["0", "1", "2", "X"]:
                    raise ValueError("Símbolo no permitido en laberinto.")
            matriz_temporal.append(fila_limpia)
            
    mapa_numpy = np.array(matriz_temporal, dtype=str)
    m, r = mapa_numpy.shape
    
    inicios = np.argwhere(mapa_numpy == "1")
    metas = np.argwhere(mapa_numpy == "2")
    if len(inicios) != 1 or len(metas) != 1:
        raise ValueError("Inicio o Meta no únicos.")
        
    inicio = tuple(inicios[0])
    meta = tuple(metas[0])
    
    # Validar perímetro de muros (X)
    if not (np.all(mapa_numpy[0, :] == "X") and np.all(mapa_numpy[m-1, :] == "X") and
            np.all(mapa_numpy[:, 0] == "X") and np.all(mapa_numpy[:, r-1] == "X")):
        raise ValueError("El perímetro debe estar amurallado con X.")
        
    # Validar posiciones de inicio y meta
    if inicio[0] != 1 or meta[0] != m - 2:
        raise ValueError("Inicio debe estar en la fila 2 y la Meta en la fila m-1 (1-based).")
        
    return mapa_numpy, inicio, meta

print("Lógica de validación cargada correctamente.")

### Validación del Vecindario de Moore
El vecindario de Moore de radio $r=1$ de una celda $(i, j)$ está constituido por las 8 celdas adyacentes:
$$V(i, j) = \{(i+di, j+dj) \mid di, dj \in \{-1, 0, 1\}, (di, dj) \neq (0, 0)\}$$

Para evitar callejones sin salida inmediatos o laberintos irresolubles, el código verifica que ninguna de estas 8 celdas contenga un muro `'X'` en las coordenadas internas transitables:

In [ ]:
def vecindario_interior_despejado(mapa_numpy, posicion):
    m, r = mapa_numpy.shape
    i, j = posicion
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            if di == 0 and dj == 0: 
                continue
            ni, nj = i + di, j + dj
            if 0 < ni < m - 1 and 0 < nj < r - 1:
                if mapa_numpy[ni, nj] == "X":
                    return False
    return True

## 4. Filosofía de la Función Objetivo $J(x)$ y Penalizaciones (`fitness.py`)

### La Heurística de Evaluación
La función objetivo $J(x)$ es una **heurística de evaluación** y no una demostración matemática de optimalidad global. En problemas evolutivos, un espacio de búsqueda sin penalizaciones intermedias sería extremadamente plano (el robot que choca contra un muro al inicio tendría el mismo costo que el que recorre el 90% del laberinto pero choca al final, rompiendo el gradiente de fitness).

Las penalizaciones actúan como una **fuerza de fricción en el paisaje de aptitud**, distorsionando el espacio de búsqueda para guiar la evolución hacia soluciones limpias y viables.

### Formulación Matemática de Penalizaciones:

1. **Distancia de Manhattan**: Penaliza la distancia ortogonal del robot a la meta al final de su simulación:
   $$D(x) = |x_{final} - x_{meta}| + |y_{final} - y_{meta}|$$
2. **Tiempo de detención efectivo $\tau(x)$**: Si el robot llega a la meta y se mantiene inactivo mediante la acción $Q$, $\tau(x)$ es el paso del primer arribo. Si no es válido (nunca llega o realiza movimientos post-meta), se penaliza con $n+1$ para castigar la inactividad.
3. **Pausas intermedias ($P_{pausas}$)**: Si realiza paradas $Q$ intermedias antes de terminar su ruta real:
   $$P_{pausas} = 10 \times \text{pausas}$$
4. **Choques contra muros ($P_{choques}$)**: Penaliza el impacto energético e ineficiencia de golpear paredes:
   $$P_{choques} = 30 \times \text{choques}$$
5. **Bloques de giros inútiles ($P_{giros}$)**: Evita ciclos redundantes en la misma coordenada:
   - Bloque de longitud 2: $10$ pts.
   - Bloque de longitud 3: $30$ pts.
   - Bloques mayores: $120 \times (L - 3)$ pts.
6. **Acciones post-meta ($P_{post\_meta}$)**: Castiga continuar moviéndose tras haber completado el laberinto:
   $$P_{post\_meta} = 100 \times \text{acciones post-meta}$$
7. **Detención prematura ($P_{prem}$)**: Penaliza terminar la cinta de comandos en reposo $Q$ si no se ha alcanzado la meta:
   $$P_{prem} = 10 \times \text{comandos } Q \text{ finales en rutas inválidas}$$
8. **Invalidez masiva**: Penalización fija de $10,000$ pts si la solución es inviable ($es\_valido = False$).

In [ ]:
# Fragmento de código de la Función Objetivo J en fitness.py
def funcion_objetivo_J(resultado):
    p_pausas = 10 * resultado.pausas_intermedias
    p_choques = 30 * resultado.choques
    
    p_giros = 0.0
    for bloque in resultado.bloques_giros:
        if bloque == 2:
            p_giros += 10
        elif bloque == 3:
            p_giros += 30
        elif bloque > 3:
            p_giros += 120 * (bloque - 3)
            
    p_post_meta = 100 * resultado.acciones_post_meta
    p_prematura = 10 * resultado.detencion_prematura
    p_invalidez = 0 if resultado.es_valido else 10000
    
    return (
        resultado.distancia_final
        + resultado.tau
        + p_pausas
        + p_choques
        + p_giros
        + p_post_meta
        + p_prematura
        + p_invalidez
    )

## 5. Reglas de Factibilidad de Deb y Selección (`seleccion.py`)

### Reglas de Factibilidad de Deb
Para optimizar funciones sujetas a restricciones, las reglas de Deb establecen que las soluciones factibles (válidas) deben **dominar estrictamente** a las soluciones inviables. 

En nuestro modelo, definimos la prioridad de factibilidad $\rho(x)$:
*   $\rho = 0$: Solución válida (llega a la meta y se detiene en ella con comandos $Q$).
*   $\rho = 1$: Solución que tocó la meta pero no finalizó en reposo (inválida).
*   $\rho = 2$: Solución que nunca alcanzó la meta.

Al ordenar con la tupla lexicográfica $(\rho, J, D, \tau)$, el algoritmo prioriza siempre la viabilidad del camino, evitando la **convergencia prematura** en mínimos locales (por ejemplo, rutas cortas que no llegan a la meta pero evitaron penalizaciones por choque).

In [ ]:
# Ordenamiento de la población según Deb
def prioridad_factibilidad(resultado):
    if resultado.es_valido:
        return 0
    if len(resultado.llegadas_efectivas) > 0:
        return 1
    return 2

def clave_ordenamiento(individuo):
    cromo, resultado = individuo
    rho = prioridad_factibilidad(resultado)
    J = funcion_objetivo_J(resultado)
    D = resultado.distancia_final
    tau = resultado.tau
    return (rho, J, D, tau)

def ordenar_poblacion(poblacion):
    return sorted(poblacion, key=clave_ordenamiento)

### Matemática del Ranking Geométrico
Para una presión selectiva $ps \in (0, 1)$, la probabilidad acumulada de seleccionar al individuo ordenado en el rango $i$ (donde el índice $i=1$ es el mejor y el índice $i=N$ es el peor) se formula como:
$$C_i = \frac{1 - (1 - ps)^i}{1 - (1 - ps)^N}$$

*   **Deriva Genética (Stochastic Drift)**: Si $ps$ es demasiado bajo, la probabilidad de reproducirse se homogeneiza y la población pierde la presión evolutiva, convirtiendo el algoritmo en una caminata aleatoria.
*   **Estancamiento**: Si $ps$ es demasiado alto, los super-individuos dominan de inmediato la población, reduciendo la diversidad genética y bloqueando la exploración de mejores soluciones en generaciones tardías.

In [ ]:
# Distribución acumulada del Ranking Geométrico
def distribucion_acumulada(N, ps):
    return [(1 - (1 - ps) ** i) / (1 - (1 - ps) ** N) for i in range(1, N + 1)]

def seleccionar_parental(poblacion_ord, ps, rng):
    N = len(poblacion_ord)
    C = distribucion_acumulada(N, ps)
    u = rng.random()  # Generar número uniforme u en [0,1)
    for i, ci in enumerate(C):
        if u <= ci:
            return poblacion_ord[i]
    return poblacion_ord[-1]

## 6. Operadores Evolutivos (`cromosoma.py` y `orquestacion.py`)

*   **Cruce Monopunto**: Intercambia material genético en un punto de corte válido $c \in [1, n-1]$:
    $$H_1 = (P_{1, :c} + P_{2, c:}), \quad H_2 = (P_{2, :c} + P_{1, c:})$$
*   **Mutación No-Nula**: Si un gen muta con probabilidad $pm$, se selecciona obligatoriamente una de las 3 alternativas restantes del alfabeto, asegurando el cambio dinámico y evitando iteraciones nulas.
*   **Elitismo Absoluto**: El mejor individuo del histórico global absoluto de la simulación se inserta intacto como el primer miembro de la población de la generación siguiente, garantizando que no se pierdan las mejores rutas descubiertas.

In [ ]:
# Ejemplo de Cruce y Mutación en cromosoma.py
class Cromosoma:
    def __init__(self, genes):
        self._genes = genes
        
    def cruzar_un_punto(self, otro, c):
        return (
            Cromosoma(self._genes[:c] + otro._genes[c:]),
            Cromosoma(otro._genes[:c] + self._genes[c:]),
        )
        
    def mutar(self, pm, rng):
        # Tabla de alternativas para asegurar mutación no-nula
        alternativas = {
            'H': ('A', 'M', 'Q'),
            'A': ('H', 'M', 'Q'),
            'M': ('H', 'A', 'Q'),
            'Q': ('H', 'A', 'M')
        }
        for i in range(len(self._genes)):
            if rng.random() < pm:
                self._genes[i] = rng.choice(alternativas[self._genes[i]])